# LoPy 影视级工作流 - Google Colab 快速运行笔记本 (MiniMax-H3-Image 评测版)

本笔记本支持在 **Google Colab** 环境下极速运行 MiniMax 最新发布的 **MiniMax-H3-Image (33.1B DiT)** 图像生成与编辑模型，以及 MiniMax H3 (Ref2VA) 视频模型。

> 💡 **核心运行建议**：
> 1. **显卡选型**：在顶部菜单【代码执行程序】->【更改运行时类型】中，**首选 G4 (RTX PRO 6000 96GB Blackwell)**，单镜出片仅需 42 秒，单片成本最低（0.10 单元）；备选 **L4 (24GB Ada Lovelace)**。
> 2. **存储架构（动静分离）**：42GB 大模型利用机房千兆网络直接高速下载至本地临时盘（零 FUSE 延迟、零 403 配额风险）；Google Drive **仅用于实时自动保存成片与输出资产**，防止因会话超时丢失成果！

In [ ]:
# [步骤 1] 检查 GPU 分配情况
!nvidia-smi

### [步骤 2] 挂载 Google Drive（仅用于实时保存成片，防意外丢失）
将输出目录 `runtime/outputs` 实时软链接到您的 Google 云端硬盘（`MyDrive/lopy_outputs`）。
每一张生成的首帧、每一条生成的视频，落盘即自动备份入云盘，即使 Colab 断线成果也绝不丢失！

In [ ]:
from google.colab import drive
import os

USE_DRIVE = True # 强烈建议开启：自动归档成片
if USE_DRIVE:
    drive.mount('/content/drive')
    drive_out = '/content/drive/MyDrive/lopy_outputs'
    os.makedirs(drive_out, exist_ok=True)
    print(f"✅ Google Drive 挂载成功！输出将实时保存至: {drive_out}")
else:
    print("跳过 Google Drive 挂载，成片将仅保存在 Colab 临时磁盘。")

### [步骤 3] 克隆仓库 (dev-ai 分支) 并一键配置环境
拉取最新的项目代码与规范工作区，并运行 `cloud/colab_setup.sh` 安装 ComfyUI 与必要算子。

In [ ]:
%cd /content
!rm -rf lopy-workflow-colab
!git clone -b dev-ai https://github.com/uljjmhn520/lopy-workflow-colab.git
%cd /content/lopy-workflow-colab

# 若开启了 Drive，建立成片输出目录的自动归档软链接
if USE_DRIVE:
    !rm -rf runtime/outputs
    !ln -sfn /content/drive/MyDrive/lopy_outputs runtime/outputs
    print("✅ runtime/outputs 已成功映射至 Google Drive！")

!bash cloud/colab_setup.sh

### [步骤 4] 极速多线程下载 MiniMax-H3-Image 模型权重
利用 Google Cloud 千兆带宽，通过 `aria2c` 直接下载到本地 NVMe 临时盘（耗时约 2~4 分钟，完全不走 Google Drive FUSE）。
- 扩散模型：`minimax_h3_image_pruned_int8_convrot.safetensors` (~21GB)
- 文本/视觉编码器：`qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors` (~15.7GB，与 H3 视频共用)
- 图像 VAE：`minimax_h3_image_vae_fp16.safetensors` (~1.17GB)

In [ ]:
from pathlib import Path

MODELS_DIR = Path("/content/lopy-workflow-colab/models/H3/models")
(MODELS_DIR / "diffusion_models").mkdir(parents=True, exist_ok=True)
(MODELS_DIR / "text_encoders").mkdir(parents=True, exist_ok=True)
(MODELS_DIR / "vae").mkdir(parents=True, exist_ok=True)

files = [
    ("diffusion_models/minimax_h3_image_pruned_int8_convrot.safetensors", "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/minimax_h3_image_pruned_int8_convrot.safetensors"),
    ("text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors", "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors"),
    ("vae/minimax_h3_image_vae_fp16.safetensors", "https://huggingface.co/Comfy-Org/MiniMax-H3-Image/resolve/main/minimax_h3_image_vae_fp16.safetensors")
]

for rel_path, url in files:
    out_path = MODELS_DIR / rel_path
    if not out_path.exists():
        print(f"[下载中] {rel_path} ...")
        !aria2c -c -x 16 -s 16 --console-log-level=warn "{url}" -d "{out_path.parent}" -o "{out_path.name}"
    else:
        print(f"[已就绪] {rel_path}")

print("✅ MiniMax-H3-Image 核心模型准备就绪！")

### [步骤 5] 后台启动 ComfyUI 推理引擎

In [ ]:
import subprocess, time, urllib.request

cmd = (
    "python3 /content/ComfyUI/main.py "
    "--listen 127.0.0.1 --port 8188 "
    "--input-directory /content/lopy-workflow-colab/runtime/inputs "
    "--output-directory /content/lopy-workflow-colab/runtime/outputs "
    "--lowvram"
)

log_file = open("/content/comfy_server.log", "w")
proc = subprocess.Popen(cmd, shell=True, stdout=log_file, stderr=subprocess.STDOUT)
print("正在启动 ComfyUI 引擎...")

# 检查服务就绪状态
for _ in range(30):
    try:
        r = urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=2)
        if r.status == 200:
            print("✅ ComfyUI 服务已成功启动！")
            break
    except Exception:
        time.sleep(2)
else:
    print("启动超时，请查看 /content/comfy_server.log 日志。")

### [步骤 6] 运行 MiniMax-H3-Image 图像测试（6步蒸馏极速出片）
生成 9:16 竖屏 (768×1344) 角色立绘/首帧测试图。在 G4/L4 上预计只需 10~25 秒。

In [ ]:
# 使用 MiniMax-H3-Image 6步快速采样
!python3 tools/gen_image.py --model minimax-image --width 768 --height 1344 \
  --prompt "1girl, Xiaoyun, East Asian high school student, neat black ponytail, wearing school uniform, sitting in classroom, sunset lighting from left window, cinematic 500T grain, warm tone, masterpiece" \
  --prefix h3_image_test

### [步骤 7] 图像预览与效果检视

In [ ]:
from IPython.display import Image, display
import glob

# 寻找最新生成的图像（已自动保存在 Google Drive 与本地 outputs 目录中）
imgs = sorted(glob.glob("/content/lopy-workflow-colab/runtime/outputs/*h3_image_test*.png"))
if imgs:
    latest_img = imgs[-1]
    print(f"最新生成图像: {latest_img}")
    display(Image(latest_img, width=450))
else:
    print("未检测到生成图像，请检查 ComfyUI 控制台日志 /content/comfy_server.log。")